In [0]:
import dlt
from pyspark.sql.functions import col, when, trim, concat_ws, to_date, coalesce, lit, current_timestamp

# HELPER FUNCTIONS

def clean_str(c):
    """Trims whitespace and converts raw '\\N' strings to SQL NULL."""
    return when(trim(col(c)) == "\\N", None).otherwise(trim(col(c)))
# 1. SILVER DRIVERS

@dlt.table(
    name="silver_drivers",
    comment="Cleaned drivers table with full names, typed DOB, and null numbers mapped to 0",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_driver_id", "driver_id IS NOT NULL")
def silver_drivers():
    return (
        dlt.read_stream("bronze_drivers")
        .select(
            clean_str("driverId").cast("int").alias("driver_id"),
            clean_str("driverRef").alias("driver_ref"),
            # Coalesce: Converts Null numbers to 0
            coalesce(clean_str("number").cast("int"), lit(0)).alias("driver_number"),
            clean_str("code").alias("driver_code"),
            clean_str("forename").alias("forename"),
            clean_str("surname").alias("surname"),
            concat_ws(" ", clean_str("forename"), clean_str("surname")).alias("driver_name"),
            to_date(clean_str("dob"), "yyyy-MM-dd").alias("date_of_birth"),
            clean_str("nationality").alias("nationality"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["driver_id"])
    )

# 2. SILVER RACES

@dlt.table(
    name="silver_races",
    comment="Cleaned core race schedules (practice and sprint dates removed)",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_race_and_circuit", "race_id IS NOT NULL AND circuit_id IS NOT NULL")
def silver_races():
    return (
        dlt.read_stream("bronze_races")
        .select(
            clean_str("raceId").cast("int").alias("race_id"),
            clean_str("year").cast("int").alias("race_year"),
            clean_str("round").cast("int").alias("race_round"),
            clean_str("circuitId").cast("int").alias("circuit_id"),
            clean_str("name").alias("race_name"),
            to_date(clean_str("date"), "yyyy-MM-dd").alias("race_date"),
            clean_str("time").alias("race_time"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["race_id"])
    )

# ------------------------------------------------------------------------------
# 3. SILVER RESULTS
# ------------------------------------------------------------------------------
@dlt.table(
    name="silver_results",
    comment="Cleaned race performance results with cast numeric metrics",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_result_fks", "result_id IS NOT NULL AND race_id IS NOT NULL AND driver_id IS NOT NULL AND constructor_id IS NOT NULL")
def silver_results():
    return (
        dlt.read_stream("bronze_results")
        .select(
            clean_str("resultId").cast("int").alias("result_id"),
            clean_str("raceId").cast("int").alias("race_id"),
            clean_str("driverId").cast("int").alias("driver_id"),
            clean_str("constructorId").cast("int").alias("constructor_id"),
            coalesce(clean_str("number").cast("int"), lit(0)).alias("driver_number"),
            clean_str("grid").cast("int").alias("grid_position"),
            clean_str("position").cast("int").alias("final_position"),
            clean_str("positionText").alias("position_text"),
            clean_str("points").cast("float").alias("points_scored"),
            clean_str("laps").cast("int").alias("laps_completed"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["result_id"])
    )

# ------------------------------------------------------------------------------
# 4. SILVER CIRCUITS
# ------------------------------------------------------------------------------
@dlt.table(
    name="silver_circuits",
    comment="Cleaned circuits location coordinates",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_circuit_id", "circuit_id IS NOT NULL")
def silver_circuits():
    return (
        dlt.read_stream("bronze_circuits")
        .select(
            clean_str("circuitId").cast("int").alias("circuit_id"),
            clean_str("circuitRef").alias("circuit_ref"),
            clean_str("name").alias("circuit_name"),
            clean_str("location").alias("location"),
            clean_str("country").alias("country"),
            clean_str("lat").cast("float").alias("latitude"),
            clean_str("lng").cast("float").alias("longitude"),
            clean_str("alt").cast("int").alias("altitude"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["circuit_id"])
    )

# ------------------------------------------------------------------------------
# 5. SILVER CONSTRUCTORS
# ------------------------------------------------------------------------------
@dlt.table(
    name="silver_constructors",
    comment="Cleaned constructors/teams metadata",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_constructor_id", "constructor_id IS NOT NULL")
def silver_constructors():
    return (
        dlt.read_stream("bronze_constructors")
        .select(
            clean_str("constructorId").cast("int").alias("constructor_id"),
            clean_str("constructorRef").alias("constructor_ref"),
            clean_str("name").alias("team_name"),
            clean_str("nationality").alias("team_nationality"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["constructor_id"])
    )


# ------------------------------------------------------------------------------
# 6. SILVER LAP TIMES
# ------------------------------------------------------------------------------
@dlt.table(
    name="silver_lap_times",
    comment="Cleaned granular lap times dataset",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_lap_entry", "race_id IS NOT NULL AND driver_id IS NOT NULL")
def silver_lap_times():
    return (
        dlt.read_stream("bronze_lap_times")
        .select(
            clean_str("raceId").cast("int").alias("race_id"),
            clean_str("driverId").cast("int").alias("driver_id"),
            clean_str("lap").cast("int").alias("lap_number"),
            clean_str("position").cast("int").alias("position"),
            clean_str("milliseconds").cast("int").alias("lap_time_ms"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates()
    )

# ------------------------------------------------------------------------------
# 7. SILVER DRIVER STANDINGS
# ------------------------------------------------------------------------------
@dlt.table(
    name="silver_driver_standings",
    comment="Cleaned driver championship standings",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_driver_standings_id", "driver_standings_id IS NOT NULL AND race_id IS NOT NULL AND driver_id IS NOT NULL")
def silver_driver_standings():
    return (
        dlt.read_stream("bronze_driver_standings")
        .select(
            clean_str("driverStandingsId").cast("int").alias("driver_standings_id"),
            clean_str("raceId").cast("int").alias("race_id"),
            clean_str("driverId").cast("int").alias("driver_id"),
            clean_str("points").cast("float").alias("points"),
            clean_str("position").cast("int").alias("position"),
            clean_str("positionText").alias("position_text"),
            clean_str("wins").cast("int").alias("wins"),
            current_timestamp().alias("silver_ingestion_timestamp")
        )
        .dropDuplicates(["driver_standings_id"])
    )